In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:

DATA_DIR = Path(".")
sales = pd.read_csv("data/raw/sales_data.csv")
future = pd.read_csv("data/raw/future_values.csv")
meta = pd.read_csv("data/raw/metadata.csv")

print("sales:", sales.shape)
print("future:", future.shape)
print("meta:", meta.shape)



sales: (624624, 8)
future: (40560, 7)
meta: (676, 4)


C:\Users\angel\AppData\Local\Temp\ipykernel_30052\3095133760.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv("data/raw/sales_data.csv")


In [7]:
# 1. Initial Data Inspection
# 查看每個資料集有哪些欄位

from IPython.display import Markdown, display

datasets = {
    "Sales": sales,
    "Future": future,
    "Metadata": meta
}

for name, df in datasets.items():

    display(Markdown(f"# {name}"))

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    display(df.head())

    print("\n" + "="*120)

# Sales

Shape: (624624, 8)

Columns:
['store_id', 'date', 'sales', 'customers', 'open', 'promo', 'state_holiday', 'school_holiday']

Data Types:
store_id          object
date              object
sales              int64
customers          int64
open               int64
promo              int64
state_holiday     object
school_holiday     int64
dtype: object


,store_id,date,sales,customers,open,promo,state_holiday,school_holiday
0,store_1,2015-07-19,0,0,0,0,0,0
1,store_2,2015-07-19,0,0,0,0,0,0
2,store_3,2015-07-19,0,0,0,0,0,0
3,store_4,2015-07-19,0,0,0,0,0,0
4,store_5,2015-07-19,0,0,0,0,0,0


# Future

Shape: (40560, 7)

Columns:
['store_id', 'date', 'open', 'promo', 'state_holiday', 'school_holiday', 'customers']

Data Types:
store_id           object
date               object
open              float64
promo               int64
state_holiday       int64
school_holiday      int64
customers         float64
dtype: object


,store_id,date,open,promo,state_holiday,school_holiday,customers
0,store_1,2015-09-17,1.0,1,0,0,NaN
1,store_2,2015-09-17,1.0,1,0,0,NaN
2,store_3,2015-09-17,1.0,1,0,0,NaN
3,store_4,2015-09-17,1.0,1,0,0,NaN
4,store_5,2015-09-17,1.0,1,0,0,NaN


# Metadata

Shape: (676, 4)

Columns:
['store_id', 'store_type', 'assortment', 'competition_distance']

Data Types:
store_id                 object
store_type               object
assortment               object
competition_distance    float64
dtype: object


,store_id,store_type,assortment,competition_distance
0,store_1,c,a,1270.0
1,store_2,a,a,14130.0
2,store_3,a,c,24000.0
3,store_4,a,a,7520.0
4,store_5,a,c,2030.0


In [10]:
# 2. Missing Value Inspection
from IPython.display import Markdown, display

datasets = {
    "Sales": sales,
    "Future": future,
    "Metadata": meta
}

for name, df in datasets.items():

    display(Markdown(f"# {name}"))

    missing = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        
    })

    # 只顯示有缺失值的欄位
    missing = missing[missing["Missing Count"] > 0]

    if missing.empty:
        print("No missing values.")
    else:
        display(missing)

    

# Sales

No missing values.


# Future

,Missing Count
open,11
customers,40560


# Metadata

,Missing Count
competition_distance,1


In [ ]:
#有缺失值，但都先不要補

#未來根本不知道會有多少客人。

Business Forecasting 第九章（Multivariate Forecasting）有講：

Exogenous variables 必須知道未來值才能拿來預測。

而 Customers：

今天不知道。明天不知道。下星期更不知道。所以公司故意留空。
意思就是：
不要把 Customers 當作 future feature。

#competition_distance      1

代表676家店裡只有一間店沒有競爭者距離。可能是沒資料、沒競爭者、新店、現在不知道。


之後我們可以判斷：

median
max
special value

哪個最合理。



In [11]:
#檢查日期是否連續
print("Start:", sales["date"].min())
print("End:  ", sales["date"].max())

Start: 2013-01-07
End:   2015-07-19


In [ ]:
# 3. Date Continuity Check  只看Historical Sales 
#結論:sales沒有缺失
sales["date"] = pd.to_datetime(sales["date"])

results = []

for store, group in sales.groupby("store_id"):
    actual_dates = pd.DatetimeIndex(group["date"].sort_values().unique())

    expected_dates = pd.date_range(
        start=actual_dates.min(),
        end=actual_dates.max(),
        freq="D"
    )

    missing_dates = expected_dates.difference(actual_dates)

    results.append({
        "store_id": store,
        "start_date": actual_dates.min(),
        "end_date": actual_dates.max(),
        "actual_days": len(actual_dates),
        "expected_days": len(expected_dates),
        "missing_days": len(missing_dates)
    })

date_check = pd.DataFrame(results)

display(date_check.head())

summary = (
    date_check["missing_days"]
    .value_counts()
    .sort_index()
    .reset_index()
)

summary.columns = ["missing_days", "number_of_stores"]

display(summary)

,store_id,start_date,end_date,actual_days,expected_days,missing_days
0,store_1,2013-01-07,2015-07-19,924,924,0
1,store_10,2013-01-07,2015-07-19,924,924,0
2,store_100,2013-01-07,2015-07-19,924,924,0
3,store_101,2013-01-07,2015-07-19,924,924,0
4,store_102,2013-01-07,2015-07-19,924,924,0


,missing_days,number_of_stores
0,0,676
